<a href="https://colab.research.google.com/github/Padinjavier/google_colab/blob/main/CODE_MUSIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS
!pip install -q gradio yt-dlp mutagen requests

import os
import re
import json
import csv
import io
import shutil
import requests
import zipfile
import gradio as gr
from mutagen.mp4 import MP4, MP4Cover

# ─────────────────────────────────────────────
# SPOTIFY SCRAPER (Extracción sin API Key)
# ─────────────────────────────────────────────
class SpotifyAPI:
    _EMBED_PLAYLIST_URL = "https://open.spotify.com/embed/playlist/{playlist_id}"
    _EMBED_TRACK_URL = "https://open.spotify.com/embed/track/{track_id}"
    _NEXT_DATA_PATTERN = re.compile(r'<script id="__NEXT_DATA__"[^>]*>([^<]+)</script>')
    _USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

    def __init__(self):
        self._session = requests.Session()

    def _fetch_embed_data(self, url: str) -> dict:
        headers = {"accept": "text/html", "user-agent": self._USER_AGENT}
        response = self._session.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}")
        match = self._NEXT_DATA_PATTERN.search(response.text)
        if not match:
            raise Exception("No se encontró __NEXT_DATA__")
        return json.loads(match.group(1))

    def _extract_entity(self, data: dict) -> dict:
        paths = [
            ("props", "pageProps", "state", "data", "entity"),
            ("props", "pageProps", "data", "entity"),
            ("props", "pageProps", "entity"),
        ]
        for path in paths:
            result = data
            for key in path:
                if isinstance(result, dict):
                    result = result.get(key)
            if isinstance(result, dict) and result:
                return result
        raise Exception("No se pudo extraer la entidad")

    def get_playlist_info(self, playlist_id: str) -> dict:
        url = self._EMBED_PLAYLIST_URL.format(playlist_id=playlist_id)
        data = self._fetch_embed_data(url)
        entity = self._extract_entity(data)

        tracks = []
        for track in entity.get("trackList", []):
            if not isinstance(track, dict): continue
            uri = track.get("uri", "")
            track_id = uri.split(":")[-1] if uri.startswith("spotify:track:") else ""
            if not track_id: continue

            title = track.get("title") or track.get("name") or "Unknown"
            artists_raw = track.get("subtitle") or track.get("artists") or ""
            if isinstance(artists_raw, list):
                artists = ", ".join(a.get("name", "") for a in artists_raw if isinstance(a, dict))
            else:
                artists = str(artists_raw)

            tracks.append({"id": track_id, "title": title, "artists": artists})

        cover_url = None
        sources = entity.get("coverArt", {}).get("sources", [])
        if sources: cover_url = sources[-1].get("url")

        return {
            "name": entity.get("name") or "Playlist_Spotify",
            "tracks": tracks,
            "cover_url": cover_url
        }

    def get_track_metadata(self, track_id: str) -> dict:
        try:
            url = self._EMBED_TRACK_URL.format(track_id=track_id)
            data = self._fetch_embed_data(url)
            entity = self._extract_entity(data)
            album = entity.get("album", {}).get("name", "")

            cover_url = None
            images = entity.get("visualIdentity", {}).get("image", [])
            if images:
                cover_url = sorted(images, key=lambda i: i.get("maxWidth", 0), reverse=True)[0].get("url")

            return {"album": album, "cover_url": cover_url}
        except Exception:
            return {"album": "", "cover_url": None}

# ─────────────────────────────────────────────
# DESCARGADOR DE YOUTUBE & METADATOS
# ─────────────────────────────────────────────
def download_single_track(title, artist, track_id, output_dir, spotify_api, default_cover=None):
    from yt_dlp import YoutubeDL

    safe_title = re.sub(r'[<>:"/\\|?*]', '', title)
    safe_artist = re.sub(r'[<>:"/\\|?*]', '', artist)
    base_name = f"{safe_title} - {safe_artist}"
    output_template = os.path.join(output_dir, f"{base_name}.%(ext)s")

    search_query = f"ytsearch1:{title} {artist} audio"
    ydl_opts = {
        "format": "bestaudio[ext=m4a]/bestaudio/best",
        "outtmpl": output_template,
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": True
    }

    with YoutubeDL(ydl_opts) as ydl:
        ydl.extract_info(search_query, download=True)

    # Buscar archivo descargado
    downloaded_file = None
    for f in os.listdir(output_dir):
        if f.startswith(base_name) and not f.endswith(('.part', '.ytdl', '.temp')):
            downloaded_file = os.path.join(output_dir, f)
            break

    if not downloaded_file:
        return None

    # Obtener Metadatos e Inyectar Carátula
    meta = spotify_api.get_track_metadata(track_id) if track_id else {"album": "", "cover_url": None}
    cover_url = meta.get("cover_url") or default_cover

    try:
        audio = MP4(downloaded_file)
        audio["\xa9nam"] = [title]
        audio["\xa9ART"] = [artist]
        if meta.get("album"): audio["\xa9alb"] = [meta["album"]]

        if cover_url:
            res = requests.get(cover_url, timeout=10)
            if res.status_code == 200:
                audio["covr"] = [MP4Cover(res.content, imageformat=MP4Cover.FORMAT_JPEG)]
        audio.save()
    except Exception as e:
        print(f"Error etiquetando {title}: {e}")

    # Renombrar a .mp3
    final_mp3 = os.path.join(output_dir, f"{base_name}.mp3")
    os.rename(downloaded_file, final_mp3)
    return final_mp3

# ─────────────────────────────────────────────
# LÓGICA DE PROCESAMIENTO
# ─────────────────────────────────────────────
def process_spotify_url(url, progress=gr.Progress()):
    if not url: return "Pega una URL válida", None

    match = re.search(r"(https:\/\/open\.spotify\.com\/)(playlist|track)\/([a-zA-Z0-9]+)", url)
    if not match: return "URL de Spotify no válida", None

    spotify_type, item_id = match.group(2), match.group(3)
    spotify_api = SpotifyAPI()

    work_dir = "/content/downloads"
    if os.path.exists(work_dir): shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    if spotify_type == "playlist":
        info = spotify_api.get_playlist_info(item_id)
        tracks = info["tracks"]
        playlist_name = re.sub(r'[<>:"/\\|?*]', '', info["name"])
        cover_url = info.get("cover_url")

        progress(0, desc=f"Descargando playlist: {playlist_name}")
        downloaded_files = []

        for idx, t in enumerate(tracks):
            progress((idx + 1) / len(tracks), desc=f"[{idx+1}/{len(tracks)}] {t['title']}")
            f = download_single_track(t["title"], t["artists"], t["id"], work_dir, spotify_api, cover_url)
            if f: downloaded_files.append(f)

        # Crear ZIP para descargar todo el álbum/playlist de un tiro
        zip_path = f"/content/{playlist_name}.zip"
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for file in downloaded_files:
                zipf.write(file, os.path.basename(file))

        return f"¡Completado! Se descargaron {len(downloaded_files)} canciones.", zip_path

    return "Procesamiento de pista individual completado", None

def process_csv_file(file_obj, progress=gr.Progress()):
    if file_obj is None: return "Sube un archivo CSV", None

    spotify_api = SpotifyAPI()
    work_dir = "/content/downloads_csv"
    if os.path.exists(work_dir): shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    # Leer CSV
    content = file_obj.decode('utf-8')
    delimiter = ',' if ',' in content.splitlines()[0] else '\t'
    reader = csv.DictReader(io.StringIO(content), delimiter=delimiter)

    tracks = []
    for row in reader:
        title = row.get('Song') or row.get('title') or row.get('name') or ''
        artist = row.get('Artist') or row.get('artist') or ''
        track_id = row.get('Track ID') or row.get('track_id') or ''
        if title:
            tracks.append({"title": title, "artist": artist, "id": track_id})

    if not tracks: return "No se encontraron canciones en el CSV", None

    downloaded_files = []
    for idx, t in enumerate(tracks):
        progress((idx + 1) / len(tracks), desc=f"[{idx+1}/{len(tracks)}] {t['title']}")
        f = download_single_track(t["title"], t["artist"], t.get("id"), work_dir, spotify_api)
        if f: downloaded_files.append(f)

    csv_name = os.path.splitext(os.path.basename(file_obj.name))[0] if hasattr(file_obj, 'name') else "Playlist_CSV"
    zip_path = f"/content/{csv_name}.zip"
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for file in downloaded_files:
            zipf.write(file, os.path.basename(file))

    return f"¡Completado! Se procesaron {len(downloaded_files)} canciones del CSV.", zip_path

# ─────────────────────────────────────────────
# INTERFAZ WEB CON GRADIO
# ─────────────────────────────────────────────
with gr.Blocks(title="CB Music - Cloud Downloader") as app:
    gr.Markdown("# 🎵 CB Music Web (Servidor Google Colab)")
    gr.Markdown("Pega un enlace de Spotify o sube un archivo CSV para descargar tu música.")

    with gr.Tab("URL de Spotify"):
        url_input = gr.Textbox(label="Enlace de Playlist o Canción", placeholder="https://open.spotify.com/playlist/...")
        btn_url = gr.Button("Descargar Spotify", variant="primary")
        out_text_url = gr.Textbox(label="Estado")
        out_file_url = gr.File(label="Descargar Archivo (ZIP/MP3)")
        btn_url.click(process_spotify_url, inputs=[url_input], outputs=[out_text_url, out_file_url])

    with gr.Tab("Subir CSV"):
        csv_input = gr.File(label="Seleccionar archivo CSV", type="binary")
        btn_csv = gr.Button("Procesar y Descargar CSV", variant="primary")
        out_text_csv = gr.Textbox(label="Estado")
        out_file_csv = gr.File(label="Descargar Archivo (ZIP)")
        btn_csv.click(process_csv_file, inputs=[csv_input], outputs=[out_text_csv, out_file_csv])

# Lanzar servidor y generar URL pública
app.launch(share=True, debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 8.8 MB/s eta 0:00:00
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b1224056039fdf570f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS Y FFMPEG
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q gradio yt-dlp mutagen requests

import os
import re
import json
import csv
import io
import shutil
import requests
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
import gradio as gr
from mutagen.mp3 import MP3
from mutagen.id3 import ID3, APIC, TIT2, TPE1, TALB, TYER
from mutagen.mp4 import MP4, MP4Cover

# ─────────────────────────────────────────────
# SPOTIFY SCRAPER (Extracción Extendida)
# ─────────────────────────────────────────────
class SpotifyAPI:
    _EMBED_PLAYLIST_URL = "https://open.spotify.com/embed/playlist/{playlist_id}"
    _EMBED_TRACK_URL = "https://open.spotify.com/embed/track/{track_id}"
    _NEXT_DATA_PATTERN = re.compile(r'<script id="__NEXT_DATA__"[^>]*>([^<]+)</script>')
    _USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

    def __init__(self):
        self._session = requests.Session()

    def _fetch_embed_data(self, url: str) -> dict:
        headers = {"accept": "text/html", "user-agent": self._USER_AGENT}
        response = self._session.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}")
        match = self._NEXT_DATA_PATTERN.search(response.text)
        if not match:
            raise Exception("No se encontró __NEXT_DATA__")
        return json.loads(match.group(1))

    def _extract_entity(self, data: dict) -> dict:
        paths = [
            ("props", "pageProps", "state", "data", "entity"),
            ("props", "pageProps", "data", "entity"),
            ("props", "pageProps", "entity"),
        ]
        for path in paths:
            result = data
            for key in path:
                if isinstance(result, dict):
                    result = result.get(key)
            if isinstance(result, dict) and result:
                return result
        raise Exception("No se pudo extraer la entidad")

    def get_playlist_info(self, playlist_id: str) -> dict:
        url = self._EMBED_PLAYLIST_URL.format(playlist_id=playlist_id)
        data = self._fetch_embed_data(url)
        entity = self._extract_entity(data)

        tracks = []
        for track in entity.get("trackList", []):
            if not isinstance(track, dict): continue
            uri = track.get("uri", "")
            track_id = uri.split(":")[-1] if uri.startswith("spotify:track:") else ""
            if not track_id: continue

            title = track.get("title") or track.get("name") or "Desconocido"
            artists_raw = track.get("subtitle") or track.get("artists") or ""
            if isinstance(artists_raw, list):
                artists = ", ".join(a.get("name", "") for a in artists_raw if isinstance(a, dict))
            else:
                artists = str(artists_raw)

            tracks.append({"id": track_id, "title": title, "artists": artists})

        cover_url = None
        sources = entity.get("coverArt", {}).get("sources", [])
        if sources: cover_url = sources[-1].get("url")

        return {
            "name": entity.get("name") or "Playlist_Spotify",
            "tracks": tracks,
            "cover_url": cover_url
        }

    def get_track_metadata(self, track_id: str) -> dict:
        try:
            url = self._EMBED_TRACK_URL.format(track_id=track_id)
            data = self._fetch_embed_data(url)
            entity = self._extract_entity(data)
            album = entity.get("album", {}).get("name", "")

            cover_url = None
            images = entity.get("visualIdentity", {}).get("image", [])
            if images:
                cover_url = sorted(images, key=lambda i: i.get("maxWidth", 0), reverse=True)[0].get("url")

            return {"album": album, "cover_url": cover_url}
        except Exception:
            return {"album": "", "cover_url": None}

spotify_api = SpotifyAPI()

# ─────────────────────────────────────────────
# DESCARGA CON CONVERSIÓN Y ETIQUETADO FFmpeg
# ─────────────────────────────────────────────
def download_track(track_info, output_dir, audio_format, audio_bitrate, default_cover=None):
    from yt_dlp import YoutubeDL

    title = track_info["title"]
    artist = track_info["artists"]
    track_id = track_info.get("id")

    safe_title = re.sub(r'[<>:"/\\|?*]', '', title)
    safe_artist = re.sub(r'[<>:"/\\|?*]', '', artist)
    base_name = f"{safe_title} - {safe_artist}"

    temp_template = os.path.join(output_dir, f"{base_name}_temp.%(ext)s")
    search_query = f"ytsearch1:{title} {artist} audio"

    # Configurar formato y bitrate de salida
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": temp_template,
        "quiet": True,
        "no_warnings": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": audio_format if audio_format in ["mp3", "m4a", "flac", "wav", "opus"] else "mp3",
            "preferredquality": audio_bitrate if audio_bitrate != "Máxima" else "320",
        }],
        "ignoreerrors": True
    }

    try:
        with YoutubeDL(ydl_opts) as ydl:
            ydl.extract_info(search_query, download=True)

        final_ext = audio_format if audio_format in ["mp3", "m4a", "flac", "wav", "opus"] else "mp3"
        final_file = os.path.join(output_dir, f"{base_name}.{final_ext}")
        temp_file = os.path.join(output_dir, f"{base_name}_temp.{final_ext}")

        if os.path.exists(temp_file):
            os.rename(temp_file, final_file)
        elif not os.path.exists(final_file):
            return None, track_info

        # Inyección de metadatos según el formato elegido
        meta = spotify_api.get_track_metadata(track_id) if track_id else {"album": "", "cover_url": None}
        cover_url = meta.get("cover_url") or default_cover

        if cover_url:
            img_data = requests.get(cover_url, timeout=10).content

            if final_ext == "mp3":
                audio = MP3(final_file, ID3=ID3)
                try: audio.add_tags()
                except Exception: pass
                audio.tags.add(TIT2(encoding=3, text=title))
                audio.tags.add(TPE1(encoding=3, text=artist))
                if meta.get("album"): audio.tags.add(TALB(encoding=3, text=meta["album"]))
                audio.tags.add(APIC(encoding=3, mime='image/jpeg', type=3, desc='Cover', data=img_data))
                audio.save()
            elif final_ext == "m4a":
                audio = MP4(final_file)
                audio["\xa9nam"] = [title]
                audio["\xa9ART"] = [artist]
                if meta.get("album"): audio["\xa9alb"] = [meta["album"]]
                audio["covr"] = [MP4Cover(img_data, imageformat=MP4Cover.FORMAT_JPEG)]
                audio.save()

        return final_file, track_info
    except Exception as e:
        print(f"Error procesando {title}: {e}")
        return None, track_info

# ─────────────────────────────────────────────
# CONTROLADORES DE INTERFAZ
# ─────────────────────────────────────────────
state_playlist_data = {"tracks": [], "cover_url": None, "name": "Playlist"}

def analyze_spotify_url(url):
    if not url:
        return "Pega una URL válida", None, gr.CheckboxGroup(choices=[], value=[]), "0/0 Canciones"

    match = re.search(r"(https:\/\/open\.spotify\.com\/)(playlist|track)\/([a-zA-Z0-9]+)", url)
    if not match:
        return "URL de Spotify inválida", None, gr.CheckboxGroup(choices=[], value=[]), "0/0 Canciones"

    spotify_type, item_id = match.group(2), match.group(3)

    if spotify_type == "playlist":
        info = spotify_api.get_playlist_info(item_id)
        state_playlist_data["tracks"] = info["tracks"]
        state_playlist_data["cover_url"] = info["cover_url"]
        state_playlist_data["name"] = info["name"]

        choices = [f"{idx+1}. {t['title']} - {t['artists']}" for idx, t in enumerate(info["tracks"])]
        count_str = f"0/{len(info['tracks'])} Canciones seleccionadas para descargar"
        return f"Playlist cargada: {info['name']}", info["cover_url"], gr.CheckboxGroup(choices=choices, value=choices), count_str
    else:
        meta = spotify_api.get_track_metadata(item_id)
        single_track = [{"id": item_id, "title": "Canción Individual", "artists": "Artista"}]
        state_playlist_data["tracks"] = single_track
        state_playlist_data["cover_url"] = meta.get("cover_url")
        state_playlist_data["name"] = "Cancion_Suelta"

        choice = [f"1. Canción Individual ({item_id})"]
        return "Pista individual encontrada", meta.get("cover_url"), gr.CheckboxGroup(choices=choice, value=choice), "1/1 Canción seleccionada"

def analyze_csv_file(file_obj):
    if file_obj is None:
        return "Sube un archivo CSV", gr.CheckboxGroup(choices=[]), "0/0 Canciones"

    content = file_obj.decode('utf-8')
    delimiter = ',' if ',' in content.splitlines()[0] else '\t'
    reader = csv.DictReader(io.StringIO(content), delimiter=delimiter)

    tracks = []
    for row in reader:
        title = row.get('Song') or row.get('title') or row.get('name') or ''
        artist = row.get('Artist') or row.get('artist') or ''
        track_id = row.get('Track ID') or row.get('track_id') or ''
        if title:
            tracks.append({"title": title, "artists": artist, "id": track_id})

    state_playlist_data["tracks"] = tracks
    state_playlist_data["cover_url"] = None
    state_playlist_data["name"] = "Playlist_CSV"

    choices = [f"{idx+1}. {t['title']} - {t['artists']}" for idx, t in enumerate(tracks)]
    return f"CSV analizado: {len(tracks)} temas encontrados.", gr.CheckboxGroup(choices=choices, value=choices), f"0/{len(tracks)} Canciones seleccionadas"

def start_download_process(selected_choices, audio_format, audio_bitrate, mode_output, max_threads, progress=gr.Progress()):
    if not selected_choices:
        return "No seleccionaste ninguna canción.", None, None

    selected_indices = [int(item.split('.')[0]) - 1 for item in selected_choices]
    tracks_to_download = [state_playlist_data["tracks"][i] for i in selected_indices]

    work_dir = "/content/downloads_pro"
    if os.path.exists(work_dir): shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    total = len(tracks_to_download)
    completed = 0
    downloaded_files = []

    progress(0, desc=f"Iniciando descarga: 0/{total}")

    # Hilos en paralelo para velocidad extrema
    with ThreadPoolExecutor(max_workers=int(max_threads)) as executor:
        futures = [
            executor.submit(
                download_track,
                t,
                work_dir,
                audio_format.lower(),
                audio_bitrate.replace(" kbps", ""),
                state_playlist_data["cover_url"]
            )
            for t in tracks_to_download
        ]

        for future in as_completed(futures):
            res_file, t_info = future.result()
            completed += 1
            progress(completed / total, desc=f"Descargando {completed}/{total}: {t_info['title']}")
            if res_file:
                downloaded_files.append(res_file)

    if not downloaded_files:
        return "Error al descargar las canciones elegidas.", None, None

    status_msg = f"¡Proceso completado! Éxito: {len(downloaded_files)}/{total} temas."

    if mode_output == "ZIP de todo el contenido":
        zip_path = f"/content/{state_playlist_data['name']}.zip"
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for f in downloaded_files:
                zipf.write(f, os.path.basename(f))
        return status_msg, zip_path, None
    else:
        # Entrega de lista de archivos sueltos
        return status_msg, None, downloaded_files

# ─────────────────────────────────────────────
# INTERFAZ DE USUARIO - GRADIO BLOCKS
# ─────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="CB Music Ultra Cloud") as app:
    gr.Markdown("# 🚀 CB Music - Cloud Downloader Ultra Pro")
    gr.Markdown("Servidor de conversión y descarga de Spotify accionado por Google Colab.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Configuración de Audio y Rendimiento")
            fmt = gr.Dropdown(["MP3", "M4A", "FLAC", "WAV", "OPUS"], value="MP3", label="Formato de Salida")
            bitrate = gr.Dropdown(["320 kbps", "256 kbps", "192 kbps", "128 kbps", "Máxima"], value="320 kbps", label="Calidad / Bitrate")
            output_mode = gr.Radio(["ZIP de todo el contenido", "Archivos Sueltos / Música por Música"], value="ZIP de todo el contenido", label="Modo de Entrega")
            threads = gr.Slider(minimum=1, maximum=8, value=4, step=1, label="Hilos Paralelos (Velocidad de Descarga)")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("🔗 Desde Spotify (URL)"):
                    url_in = gr.Textbox(label="URL de Spotify", placeholder="https://open.spotify.com/playlist/...")
                    btn_analyze = gr.Button("🔍 Analizar Enlace / Ver Lista", variant="secondary")

                with gr.Tab("📄 Desde Archivo CSV"):
                    csv_in = gr.File(label="Subir Archivo CSV de Chosic/Spotify", type="binary")
                    btn_csv_analyze = gr.Button("🔍 Analizar Archivo CSV", variant="secondary")

            status_txt = gr.Textbox(label="Estado del Servidor", interactive=False)
            counter_lbl = gr.Label(value="0/0 Canciones", label="Contador de Progreso / Selección")
            cover_img = gr.Image(label="Carátula / Arte de Tapa", height=180)

    gr.Markdown("---")
    gr.Markdown("### 📋 Selección Individual de Canciones a Descargar")

    chk_tracks = gr.CheckboxGroup(label="Listado de Pistas (Desmarca las que NO quieras descargar)")
    btn_download = gr.Button("⚡ Iniciar Descarga de Pistas Seleccionadas", variant="primary")

    gr.Markdown("### 📥 Resultados y Descarga Directa a tu Dispositivo")
    out_zip = gr.File(label="Descarga Paquete Completo (ZIP)")
    out_files = gr.Files(label="Descargar Músicas Individuales / Sueltas")

    # Binds y Eventos
    btn_analyze.click(analyze_spotify_url, inputs=[url_in], outputs=[status_txt, cover_img, chk_tracks, counter_lbl])
    btn_csv_analyze.click(analyze_csv_file, inputs=[csv_in], outputs=[status_txt, chk_tracks, counter_lbl])

    btn_download.click(
        start_download_process,
        inputs=[chk_tracks, fmt, bitrate, output_mode, threads],
        outputs=[status_txt, out_zip, out_files]
    )

app.launch(share=True, debug=True)